In [ ]:
%pip install langchain langchain-text-splitters PyPDF2

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd
from PyPDF2 import PdfReader
from io import BytesIO
from snowflake.snowpark import types as T
from snowflake.snowpark.files import SnowflakeFile
from snowflake.snowpark.types import StringType
from snowflake.snowpark import Session
from snowflake.snowpark.types import StringType, StructField, StructType

In [2]:
session = Session.builder.getOrCreate()

In [4]:
session.use_database('RAG_DB')
session.use_schema('PUBLIC')

In [3]:
session.sql("create stage if not exists RAG DIRECTORY = ( ENABLE = TRUE)").collect()

In [4]:
session.file.put("documents/*.pdf", "@RAG", auto_compress=False, overwrite=True)

In [5]:
def readpdf(file_path):
    whole_text = ""
    with SnowflakeFile.open(file_path, "rb") as file:
        f = BytesIO(file.readall())
        pdf_reader = PdfReader(f)
        whole_text = ""
        for page in pdf_reader.pages:
            whole_text += page.extract_text()
    return whole_text

In [6]:
# Register the UDF.
session.udf.register(
    func=readpdf,
    return_type=StringType(),
    input_types=[StringType()],
    is_permanent=True,
    name="SNOWPARK_PDF",
    replace=True,
    packages=["snowflake-snowpark-python", "pypdf2", "PyCryptodome"],
    stage_location="RAG",
)

In [7]:
session.sql(
    """
CREATE OR REPLACE TABLE RAW_TEXT AS
SELECT
    relative_path
    , file_url
    , snowpark_pdf(build_scoped_file_url(@RAG, relative_path)) as raw_text
from directory(@RAG)
            """
).collect()

In [8]:
session.table("RAW_TEXT").show()

In [9]:
class text_chunker:

    def process(self, text):
        text_raw = []
        text_raw.append(text)

        text_splitter = RecursiveCharacterTextSplitter(
            separators=[
                "\n"
            ],  # Define an appropriate separator. New line is good typically!
            chunk_size=500,  # Adjust this as you see fit
            chunk_overlap=100,  # This let's text have some form of overlap. Useful for keeping chunks contextual
            length_function=len,
            add_start_index=True,  # Optional but useful if you'd like to feed the chunk before/after
        )

        chunks = text_splitter.create_documents(text_raw)
        df = pd.DataFrame(
            [(doc.page_content, str(doc.metadata)) for doc in chunks],
            columns=["chunks", "meta"],
        )

        yield from df.itertuples(index=False, name=None)

In [10]:
# Register the UDTF

schema = StructType(
    [
        StructField("chunk", StringType()),
        StructField("meta", StringType()),
    ]
)

session.udtf.register(
    handler=text_chunker,
    output_schema=schema,
    input_types=[StringType()],
    is_permanent=True,
    name="CHUNK_TEXT",
    replace=True,
    packages=["pandas", "langchain-text-splitters"],
    stage_location="RAG",
)

In [11]:
session.sql(
    """
CREATE OR REPLACE TABLE CHUNK_TEXT AS
SELECT
        relative_path,
        func.*
    FROM raw_text AS raw,
         TABLE(chunk_text(raw_text)) as func;
            """
).collect()

Move on to the .sql file or you can write these in a Snowflake Notebook

In [6]:
session.sql("CREATE STAGE IF NOT EXISTS RAG_IMAGES").collect()
session.file.put("images/*", "@rag_images")